# [실습] 멀티 에이전트


멀티 에이전트는 단일 에이전트의 컨텍스트 한계를, 작업을 격리해 넘어서는 구성입니다.  

대부분의 문제는 단일 에이전트로 풀 수 있습니다.  
그렇지만, 도구와 MCP/대화 메시지가 길어져 컨텍스트가 급격히 커지면 에이전트의 성능이 떨어질 수 있습니다.  

이를 해결하기 위해 세 가지 상황을 멀티 에이전트로 구현해 보겠습니다.

| 한계 상황 | 해법 패턴 |
|---|---|
| 원본 데이터가 컨텍스트에 부담되는 경우 | 서브에이전트 (원문은 격리하고 요약만 메인 에이전트에 전달) |
| 성격이 크게 다른 작업을 처리해야 하는 경우 | 슈퍼바이저 (동적 라우팅) |
| 사용자 대화 도중에 에이전트를 바꿔야 하는 경우 | 핸드오프 (단방향 인계) |

## 환경 설정

In [ ]:
%pip install langchain langchain-openai langchain-tavily langchain-community beautifulsoup4 matplotlib pyyaml -q

In [ ]:
import os, time, glob
from collections import Counter
from dotenv import load_dotenv

load_dotenv('.env', override=True)

assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY가 필요합니다 (.env)'
print('TAVILY_API_KEY:', '설정됨' if os.environ.get('TAVILY_API_KEY') else '없음 (web_search 폴백)')

from langchain.chat_models import init_chat_model


llm = init_chat_model('gpt-5.2', model_provider='openai')
print('환경 설정 완료:', type(llm).__name__)

In [ ]:
# 산출물 경로: 차트와 생성 이미지를 각각 다른 폴더에 저장합니다.
os.makedirs('outputs/charts', exist_ok=True)
os.makedirs('outputs/generated_images', exist_ok=True)
print('출력 디렉토리 준비 완료')

---
## 도구와 스킬

세 패턴이 공통으로 사용하는 도구와 스킬을 먼저 준비합니다.

| 도구 | 종류 | 비고 |
|---|---|---|
| `web_search` (tavily) | 외부 read | tools.py의 표준 도구 재사용 |
| `fetch_url` | 외부 read | 페이지 전문 가져오기 |
| `python_repl` | 코드 실행 | 수치 계산 + matplotlib 차트 실행 |
| `generate_image` | 이미지 생성 | gpt-image-2로 컨셉 이미지 생성 |
| `read_file` / `write_file` | 로컬 IO | |
| `load_skill` | 스킬 로딩 | 스킬 미들웨어가 자동 등록 |

In [ ]:
import io, contextlib
from langchain_core.tools import tool
from langchain_community.document_loaders import WebBaseLoader

from tools import web_search as tavily_search  # tools.py의 Tavily 검색 재사용


@tool
def fetch_url(url: str) -> str:
    """URL의 웹페이지 본문을 가져옵니다. url: http(s):// 웹페이지 주소"""
    if not url.startswith(('http://', 'https://')):
        return f'오류: http(s):// URL만 지원합니다. 받은 값: {url}'
    try:
        docs = WebBaseLoader(url).load()
        content = docs[0].page_content if docs else '내용을 가져올 수 없습니다.'
        return content[:5000]
    except Exception as e:
        return f'fetch_url 오류: {type(e).__name__}: {e}'


@tool
def python_repl(code: str) -> str:
    """Python 코드를 실행하고 표준출력을 반환합니다. matplotlib 차트 생성에도 사용합니다.

    Args:
        code: 실행할 파이썬 코드
    """
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {'__builtins__': __builtins__}, {})
        return buf.getvalue().strip() or '(완료, 출력 없음)'
    except Exception as e:
        return f'실행 오류: {type(e).__name__}: {e}'


@tool
def read_file(file_path: str) -> str:
    """파일 내용을 읽어 반환합니다. file_path: 파일 경로"""
    try:
        text = open(file_path, encoding='utf-8').read()
        return text[:10000] + ('\n... (생략)' if len(text) > 10000 else '')
    except Exception as e:
        return f'파일 읽기 오류: {e}'


@tool
def write_file(file_path: str, content: str) -> str:
    """파일에 내용을 작성합니다. file_path: 경로, content: 내용"""
    try:
        d = os.path.dirname(file_path)
        if d:
            os.makedirs(d, exist_ok=True)
        open(file_path, 'w', encoding='utf-8').write(content)
        return f'파일 작성 완료: {file_path} ({len(content)}자)'
    except Exception as e:
        return f'파일 작성 오류: {e}'


@tool
def generate_image(prompt: str) -> str:
    """텍스트 프롬프트로 표지/삽화 이미지를 생성해 outputs/generated_images/에 저장합니다.

    데이터 차트가 아니라 표지/삽화 같은 생성형 이미지가 필요할 때 사용합니다.
    내부적으로 gpt-image-2 모델을 호출합니다.

    Args:
        prompt: 생성할 이미지를 설명하는 텍스트. 구체적으로 작성할수록 필요한 장면을 명확히 전달할 수 있습니다.
    """
    import base64
    from datetime import datetime
    from langchain_openai import ChatOpenAI

    vis_llm = ChatOpenAI(model='gpt-5.2')
    image_tool = {'type': 'image_generation', 'model': 'gpt-image-2'}
    ai = vis_llm.bind_tools([image_tool]).invoke(prompt)
    block = next(b for b in ai.content_blocks if b['type'] == 'image')

    ts = datetime.now().strftime('%Y%m%d_%H%M%S_%f')[:-3]
    out_path = f'outputs/generated_images/image_{ts}.png'
    with open(out_path, 'wb') as f:
        f.write(base64.b64decode(block['base64']))
    return f'이미지 저장: {out_path}'


print('도구 정의 완료: web_search, fetch_url, python_repl, read_file, write_file, generate_image')


### 스킬 2종: data-viz / image-gen

Visualizer 에이전트가 쓸 스킬을 정의합니다.  
`SkillMiddleware`가 `skills/<name>/SKILL.md`를 읽어, 요약 목록은 시스템 프롬프트에 주입하고 상세 본문은 `load_skill` 도구로 그때그때 로드합니다.

- data-viz: 수치 데이터를 matplotlib 코드로 차트화 (python_repl로 실행, outputs/charts/에 저장)
- image-gen: 표지/삽화 이미지를 생성 (generate_image로 gpt-image-2 호출)

멀티 에이전트에서는 스킬도 에이전트별로 배분합니다.  
폴더 전체를 통째로 주지 않고 필요한 스킬을 명시해 `SkillMiddleware(부분집합)`로 넘기며, 여기서는 Visualizer에만 두 스킬을 채우고 나머지는 빈 미들웨어로 둡니다.

In [ ]:
from pathlib import Path
import platform

import yaml
from langchain_core.tools import tool
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.messages import SystemMessage

SKILLS_DIR = Path("skills")


def scan_skills(skills_dir: Path = SKILLS_DIR) -> dict[str, dict]:
    """SKILL.md의 YAML frontmatter를 파싱해 {name: {...}} 를 반환합니다."""
    skills: dict[str, dict] = {}
    if not skills_dir.exists():
        return skills
    for d in sorted(skills_dir.iterdir()):
        f = d / "SKILL.md"
        if not d.is_dir() or not f.exists():
            continue
        content = f.read_text(encoding="utf-8")
        if not content.startswith("---"):
            continue
        parts = content.split("---", 2)
        if len(parts) < 3:
            continue
        meta = yaml.safe_load(parts[1])
        name = meta.get("name", d.name)
        skills[name] = {
            "name": name,
            "description": meta.get("description", ""),
            "path": str(d),
            "folder": d.name,
            "meta": meta,
            "body": parts[2].strip(),
        }
    return skills


class SkillMiddleware(AgentMiddleware):
    """에이전트별 스킬 부분집합을 주입하고 바인딩합니다. 스킬셋이 비면 아무 일도 하지 않습니다.

    사용 예 (에이전트마다 다른 스킬을 명시적으로 배분):
        viz_skills = {k: SKILLS[k] for k in ['data-viz', 'image-gen']}
        visualizer = create_agent(..., middleware=[SkillMiddleware(viz_skills)])
        researcher = create_agent(..., middleware=[SkillMiddleware()])   # 빈칸 = 효과 없음
    """

    def __init__(self, skills: dict | None = None):
        # 받은 부분집합만 보유합니다.
        self.skills = dict(skills) if skills else {}
        # 빈 스킬셋이면 load_skill 도구를 등록하지 않습니다.
        self.tools = [self._make_load_skill()] if self.skills else []
        if self.skills:
            lines = []
            for name, s in self.skills.items():
                t = s["meta"].get("metadata", {}).get("type", "")
                lines.append(f"- {name}{f' [{t}]' if t else ''}: {s['description']}")
            self.skills_prompt = "\n".join(lines)
        else:
            self.skills_prompt = ""

    def _make_load_skill(self):
        """이 미들웨어가 보유한 스킬셋에 바인딩된 load_skill 도구를 만듭니다."""
        skills = self.skills  # 클로저로 부분집합을 캡처합니다.

        @tool
        def load_skill(skill_name: str) -> str:
            """스킬의 상세 본문(SKILL.md body)을 로드합니다. 해당 도메인 작업 전에 호출하세요.

            이 에이전트에 배정된 스킬만 로드할 수 있습니다.

            Args:
                skill_name: 로드할 스킬 이름
            """
            if skill_name not in skills:
                return f"스킬 '{skill_name}'을 찾을 수 없습니다. 사용 가능: {', '.join(skills) or '(없음)'}"
            skill = skills[skill_name]
            body = skill["body"]
            skill_dir = Path(skill["path"])
            # CLI 도구 스킬용: 실행 가능한 .py 스크립트 첨부 (프롬프트 묶음에는 없어 자동 생략)
            scripts = [f.name for f in skill_dir.iterdir() if f.is_file() and f.suffix == ".py"]
            if scripts:
                py = ".venv\\Scripts\\python" if platform.system() == "Windows" else ".venv/bin/python"
                body += "\n\n## 실행 가능한 스크립트\n"
                for s in scripts:
                    body += f"- `{skill['folder']}/{s}` -> `{py} skills/{skill['folder']}/{s}`\n"
            # 참고 파일(레벨 3): SKILL.md/스크립트 외 문서. 필요할 때 read_file로 읽도록 안내합니다.
            ref = [
                f"skills/{skill['folder']}/{f.relative_to(skill_dir)}".replace("\\", "/")
                for f in sorted(skill_dir.rglob("*"))
                if f.is_file() and f.name != "SKILL.md" and f.suffix != ".py"
            ]
            if ref:
                body += "\n\n## 참고 파일\n필요할 때 read_file로 읽으세요. 작업에 필요한 파일만 로드합니다.\n"
                for rf in ref:
                    body += f"- `{rf}`\n"
            return body

        return load_skill

    def _inject(self, request: ModelRequest) -> ModelRequest:
        # 스킬셋이 비면 시스템 프롬프트를 건드리지 않습니다.
        if not self.skills:
            return request
        addendum = (
            f"\n\n## Available Skills (SKILL.md 기반)\n\n{self.skills_prompt}\n\n"
            "작업 전에 `load_skill` 도구로 해당 스킬의 상세 지침을 먼저 로드하세요."
        )
        orig = request.system_message
        new_text = (orig.text + addendum) if orig else addendum
        return request.override(system_message=SystemMessage(content=new_text))

    def wrap_model_call(self, request, handler):
        return handler(self._inject(request))

    async def awrap_model_call(self, request, handler):
        return await handler(self._inject(request))


SKILLS = scan_skills()
print(f'스킬 {len(SKILLS)}개 감지:', list(SKILLS))
print('SkillMiddleware 준비 완료')

---
## 패턴 1: 서브에이전트 (여러 출처 리서치 종합)

여러 출처를 조사해 비교 정리하는 작업을 다뤄 보겠습니다.  
읽을 출처가 많아 메인 에이전트의 컨텍스트에 다 담기 어려울 때, 출처마다 서브에이전트로 나눕니다.  
각 출처는 격리된 서브에이전트(Reader)가 읽고, 필요한 사실과 인용만 압축한 요약을 돌려줍니다.  
메인 에이전트(합성자)는 원문을 보지 않고 요약만 모아 최종 리포트를 작성하므로, 원문은 경계 밖에 격리되고 경계 안으로는 요약만 넘어옵니다.

```
[Main 합성자] --read_source(subtopic)--> [Reader 서브에이전트] (격리된 윈도우에서 web_search+fetch_url)
       ^                                              |
       +--------------- 요약 + 인용 ------------------+   (원문은 Reader 안에 유지)
```

소스를 3개에서 6개로 늘려도 메인이 받는 텍스트의 양이 평탄하게 유지되는지를 측정해 보겠습니다.

### 1-1. Reader 서브에이전트를 도구로 래핑

In [ ]:
from langchain.agents import create_agent
from langchain.messages import ToolMessage, AIMessage, HumanMessage

# 조사할 서브토픽 풀 (LLM 에이전트 메모리 기법). 앞에서부터 N개를 잘라 씁니다.
SUBTOPICS = [
    'LangChain 또는 LangGraph 에이전트 short-term memory checkpointer',
    'Mem0 long-term memory for LLM agents',
    'Zep / Graphiti temporal knowledge graph agent memory',
    'Anthropic Claude memory tool context management',
    'OpenAI Assistants/ChatGPT memory feature',
    'episodic vs semantic memory for LLM agents',
    'summarization-based conversation memory compression',
]

# Reader: 한 출처를 격리된 윈도우에서 읽고 요약만 반환하는 서브에이전트입니다.
reader = create_agent(
    model=llm,
    tools=[tavily_search, fetch_url],
    system_prompt=(
        '당신은 한 가지 출처를 조사하는 리서치 서브에이전트입니다.\n'
        '내부에서 web_search(max_results=1~2)와 fetch_url을 자유롭게 쓰되, '
        '최종 응답은 반드시 아래 형식의 압축된 요약만 반환합니다.\n'
        '- 주요 사실 5개 이내, 항목당 40자 이내\n'
        '- 원문 검색 결과를 복사하지 말 것\n'
        "- 마지막 줄에 'SOURCE: <URL>' 형식으로 출처 1~2개를 표기"
    ),
)


@tool
def read_source(subtopic: str) -> str:
    """한 서브토픽을 전담 리서치 서브에이전트에 위임해 요약을 받습니다.

    원문 검색 결과는 서브에이전트의 격리된 컨텍스트에 갇히고, 호출자에게는 요약만 돌아옵니다.

    Args:
        subtopic: 조사할 단일 서브토픽
    """
    r = reader.invoke({'messages': [{'role': 'user', 'content': subtopic}]})
    return r['messages'][-1].text


print('Reader 서브에이전트를 read_source 도구로 래핑 완료')

### 1-2. Main 합성자 + 측정 헬퍼

In [ ]:
def metrics(result):
    """Main(또는 단일)이 받아들인 텍스트/토큰을 측정합니다.

    - tool_chars: 받은 tool result(요약 또는 원문)의 누적 글자수 (결정적)
    - peak_input: LLM 한 턴이 받은 최대 입력 토큰 (윈도우가 가장 커진 시점)
    """
    msgs = result['messages']
    tool_chars = sum(len(m.text or '') for m in msgs if isinstance(m, ToolMessage))
    inputs = [m.usage_metadata['input_tokens'] for m in msgs
              if isinstance(m, AIMessage) and getattr(m, 'usage_metadata', None)]
    return {
        'tool_chars': tool_chars,
        'peak_input': max(inputs, default=0),
        'turns': len(inputs),
    }


def run_multi(n):
    """Main(합성자)이 N개 서브토픽을 각각 read_source로 위임하고 요약만 모아 종합합니다."""
    subtopics = SUBTOPICS[:n]
    main = create_agent(
        model=llm,
        tools=[read_source],
        system_prompt=(
            '당신은 여러 출처 리서치 종합 에이전트입니다.\n'
            '주어진 각 서브토픽마다 read_source 도구를 호출해 요약을 모으세요.\n'
            '원문은 보지 않고 요약만 받아, 출처를 비교/종합한 한국어 리포트를 작성합니다.\n'
            '리포트에는 각 항목의 출처 URL 인용을 반드시 포함하세요.'
        ),
    )
    listing = '\n'.join(f'- {s}' for s in subtopics)
    result = main.invoke(
        {'messages': [{'role': 'user', 'content':
            f'다음 {n}개 서브토픽을 각각 read_source로 조사하고 비교 리포트를 작성해줘:\n{listing}'}]},
        config={'recursion_limit': 40},
    )
    m = metrics(result)
    m['report'] = result['messages'][-1].text
    return m


def run_mono(n):
    """모놀리식 비교 기준: 단일 에이전트가 원문 검색 결과를 전부 자기 컨텍스트에 적재합니다."""
    subtopics = SUBTOPICS[:n]
    agent = create_agent(
        model=llm,
        tools=[tavily_search, fetch_url],
        system_prompt=(
            '당신은 만능 리서치 에이전트입니다. 주어진 각 서브토픽을 직접 '
            'web_search(max_results=1~2)로 조사하고, 모든 결과를 종합해 '
            '출처 인용을 포함한 한국어 비교 리포트를 작성하세요.'
        ),
    )
    listing = '\n'.join(f'- {s}' for s in subtopics)
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content':
            f'다음 {n}개 서브토픽을 각각 조사하고 비교 리포트를 작성해줘:\n{listing}'}]},
        config={'recursion_limit': 40},
    )
    m = metrics(result)
    m['report'] = result['messages'][-1].text
    return m


print('run_multi / run_mono / metrics 준비 완료')

### 1-3. 소스 3, 6 규모별 측정

서브에이전트(멀티)와 모놀리식 단일 에이전트를 각각 N=3, 6으로 실행해, 메인 에이전트의 컨텍스트 크기가 어떻게 달라지는지 비교해 보겠습니다.  
실제 웹 검색이 일어나므로 수십 초에서 1분가량 걸립니다.

In [ ]:
Ns = [3, 6]
multi, mono = {}, {}

print('[멀티] 서브에이전트 구성, 메인은 요약만 수신합니다.')
for n in Ns:
    multi[n] = run_multi(n)
    print(f'  N={n:2d}: peak_input={multi[n]["peak_input"]:>6,}  tool_chars={multi[n]["tool_chars"]:>7,}')

print('[모놀리식] 단일 에이전트, 원문을 전부 적재합니다.')
for n in Ns:
    mono[n] = run_mono(n)
    print(f'  N={n:2d}: peak_input={mono[n]["peak_input"]:>6,}  tool_chars={mono[n]["tool_chars"]:>7,}')

In [ ]:
print('=== 성공 기준 측정 ===\n')
print('소스 수 | 멀티 메인 윈도우 | 모놀리식 윈도우 | 격리 비율')
print('-' * 60)
for n in Ns:
    ratio = mono[n]['tool_chars'] / max(multi[n]['tool_chars'], 1)
    print(f'  N={n:2d}   | {multi[n]["tool_chars"]:>10,}자   | {mono[n]["tool_chars"]:>10,}자   | {ratio:>5.1f}x')

print('\n[기준 1] 소스가 늘어도 메인 윈도우가 평탄하게 유지됩니다.')
print(f'  멀티는 소스를 {Ns[0]}개에서 {Ns[-1]}개로 늘려도 메인이 받는 텍스트가 작게 유지됩니다 '
      f'({multi[Ns[0]]["tool_chars"]:,}자에서 {multi[Ns[-1]]["tool_chars"]:,}자).')
print(f'  반면 모놀리식은 같은 구간에서 원문이 누적되어 크게 늘어납니다 '
      f'({mono[Ns[0]]["tool_chars"]:,}자에서 {mono[Ns[-1]]["tool_chars"]:,}자).')
print('\n[기준 2] 모놀리식보다 메인이 받는 토큰이 뚜렷하게 작습니다.')
print(f'  N={Ns[-1]} 기준 메인 peak_input은 멀티 {multi[Ns[-1]]["peak_input"]:,}, '
      f'모놀리식 {mono[Ns[-1]]["peak_input"]:,}로 '
      f'약 {mono[Ns[-1]]["peak_input"]/max(multi[Ns[-1]]["peak_input"],1):.1f}배 차이가 납니다.')

### 1-4. 최종 리포트 (성공 기준 3: 출처 인용 포함)

In [ ]:
# 요약만 모았는데도 각 출처의 URL 인용이 리포트에 남아 있는지 확인합니다.
report = multi[Ns[-1]]['report']
print('출처(http) 인용 포함 여부:', ('http' in report))
print('=' * 60)
print(report)

---
## 패턴 2: 슈퍼바이저 (데이터 인사이트 리포트)

성격이 다른 작업이 한 요청에 섞여 있을 때 쓰는 패턴입니다.  
Coordinator가 요청을 보고 어떤 전문가를 부를지 실행 중에 결정하며, 전문가마다 역할과 도구가 다릅니다.  
경계 안으로는 Coordinator가 내려보내는 작업 지시와 전문가가 돌려주는 구조화된 결과만 오갑니다.

| 전문가 | 도구 | 스킬 | 역할 |
|---|---|---|---|
| Coordinator | (없음) | 작업 분해 | 어떤 전문가를 부를지 결정하고 취합 |
| Researcher | `web_search`, `fetch_url` | 출처 검증 | 사실/수치 수집과 인용 |
| Quant | `python_repl` | 통계와 단위 | 수치 계산/집계, 표 결과 |
| Visualizer | `python_repl`, `generate_image` | `data-viz`, `image-gen` | 차트는 코드로, 표지 이미지는 gpt-image-2로 |

워커는 컨텍스트가 격리되어 instruction만 받습니다.  
요청에 따라 호출되는 전문가가 달라지는지, Visualizer가 데이터 차트와 표지 이미지를 상황에 맞게 구분하는지를 확인해 보겠습니다.

### 2-1. 역할이 다른 전문가 3종 (Visualizer는 스킬 보유)

In [ ]:
from langchain.agents import create_agent, AgentState
from langchain.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from pydantic import BaseModel, Field
from typing import Literal
from typing_extensions import NotRequired

WORKER_NAMES = ['researcher', 'quant', 'visualizer']

# 에이전트별 스킬 배분: 폴더 전체(SKILLS)를 통째로 주지 않고, 필요한 스킬을 명시합니다.
# 모든 전문가에 SkillMiddleware를 달아두고, 스킬을 채운 것은 visualizer뿐입니다.
VIS_SKILLS = {k: SKILLS[k] for k in ['data-viz', 'image-gen'] if k in SKILLS}

researcher = create_agent(
    model=llm,
    tools=[tavily_search, fetch_url],
    system_prompt='당신은 리서치 전문가입니다. 웹 검색과 URL 스크래핑으로 사실/수치를 수집하고 출처를 답니다. '
                  'web_search는 최대 2회, fetch_url은 최대 1회만 쓰고 필요한 사실만 간결히 정리하세요.',
    middleware=[SkillMiddleware()],   # 빈 스킬셋: 주입도 load_skill도 없음
)
quant = create_agent(
    model=llm,
    tools=[python_repl],
    system_prompt='당신은 통계/수치 계산 전문가입니다. python_repl로 계산/집계하고 결과를 표 형태로 정리합니다. '
                  'python_repl 호출은 최대 3회 이내로 끝내세요.',
    middleware=[SkillMiddleware()],   # 빈 스킬셋
)
visualizer = create_agent(
    model=llm,
    tools=[python_repl, generate_image],
    system_prompt=(
        '당신은 시각화 전문가입니다. 두 가지 산출물을 명확히 구분하세요.\n'
        "- 정확한 수치를 보여주는 데이터 '차트'는 data-viz 스킬을 로드해 python_repl로 코드 실행 "
        '(outputs/charts/에 저장).\n'
        "- 분위기/표지 같은 '컨셉 이미지'는 image-gen 스킬을 로드해 generate_image 도구로 생성 "
        '(matplotlib로 그리지 말 것).\n'
        '작업 전 해당 스킬을 load_skill로 먼저 로드하세요. '
        'generate_image가 돌려준 outputs/generated_images/... 경로를 사용하고, 그 파일을 '
        'outputs/charts/ 등 다른 폴더로 복사하거나 python_repl로 다시 저장하지 마세요(차트만 charts/에 둡니다). '
        '최종 응답에는 차트 경로(outputs/charts/...)와 생성 이미지 경로(outputs/generated_images/...)를 각각 명시하세요.'
    ),
    middleware=[SkillMiddleware(VIS_SKILLS)],   # data-viz / image-gen 만 배정
)

print('전문가 3종 정의 완료. 에이전트별 배정 스킬:')
print('  researcher:', list(SkillMiddleware().skills) or [],
      '| quant:', list(SkillMiddleware().skills) or [],
      '| visualizer:', list(VIS_SKILLS))

### 2-2. Coordinator + 동적 라우팅 그래프

In [ ]:
class CoordDecision(BaseModel):
    """Coordinator의 라우팅 결정 (구조화 출력)"""
    reasoning: str = Field(description='이 전문가를 고른 근거')
    next_agent: Literal['researcher', 'quant', 'visualizer', 'FINISH']
    instruction: str = Field(description='다음 전문가에게 줄 구체적 지시(목표/입력/출력형식)')


class SupervisorState(AgentState):
    next: NotRequired[str]


COORD_PROMPT = f"""당신은 데이터 인사이트 리포트를 만드는 작업 관리자(Coordinator)입니다.
사용 가능한 전문가: {WORKER_NAMES}

[전문가 권한]
- researcher: web_search, fetch_url (사실/수치 수집과 인용)
- quant: python_repl (수치 계산/집계, 표 형태 결과)
- visualizer: python_repl(차트), generate_image(컨셉 이미지) (data-viz/image-gen 스킬 보유)

[라우팅 규칙]
- 요청에 실제로 필요한 전문가만 부릅니다. 불필요한 전문가는 호출하지 않습니다.
- 차트나 표지/컨셉 이미지가 필요 없으면 visualizer를 부르지 않습니다.
- 수치 계산이 필요 없으면 quant를 부르지 않습니다.
- 워커는 컨텍스트가 격리되어 instruction만 받습니다. 목표/입력/출력형식을 모두 적으세요.
- 더 부를 전문가가 없으면 FINISH."""

coordinator = create_agent(
    model=llm, tools=[], system_prompt=COORD_PROMPT, response_format=CoordDecision,
)


async def coordinator_node(state) -> Command[Literal['researcher', 'quant', 'visualizer', '__end__']]:
    result = await coordinator.ainvoke({'messages': state['messages']})
    d = result['structured_response']
    print(f'  [Coordinator] -> {d.next_agent}: {d.reasoning[:60]}')
    if d.next_agent == 'FINISH':
        return Command(goto=END, update={'next': 'FINISH'})
    return Command(
        goto=d.next_agent,
        update={'next': d.next_agent,
                'messages': [HumanMessage(content=f'[지시] {d.instruction}', name='coordinator')]},
    )


def _instruction(state):
    """워커는 Coordinator의 마지막 지시만 봅니다 (컨텍스트 격리)."""
    for m in reversed(state['messages']):
        if isinstance(m, HumanMessage) and getattr(m, 'name', '') == 'coordinator':
            return m.text
    return state['messages'][0].text


def make_worker(name, agent):
    async def node(state):
        instr = _instruction(state)
        t0 = time.time()
        r = await agent.ainvoke({'messages': [HumanMessage(content=instr)]},
                                config={'recursion_limit': 30})
        print(f'  [{name}] 완료 ({time.time()-t0:.0f}s)')
        return {'messages': [HumanMessage(content=r['messages'][-1].text, name=name)]}
    return node


b2 = StateGraph(SupervisorState)
b2.add_node('coordinator', coordinator_node)
b2.add_node('researcher', make_worker('researcher', researcher))
b2.add_node('quant', make_worker('quant', quant))
b2.add_node('visualizer', make_worker('visualizer', visualizer))
b2.add_edge(START, 'coordinator')
for w in WORKER_NAMES:
    b2.add_edge(w, 'coordinator')  # 워커는 끝나면 Coordinator로 복귀합니다.
supervisor_graph = b2.compile()


def worker_dist(result):
    return Counter(getattr(m, 'name', None) for m in result['messages']
                   if getattr(m, 'name', None) in WORKER_NAMES)


supervisor_graph

### 2-3. 동적 라우팅: 요청에 따라 전문가 부분집합이 달라진다

같은 그래프에 성격이 다른 두 요청을 전달합니다.  
수치 계산만 필요한 요청 A는 `{researcher, quant}`, 차트와 표지 이미지가 필요한 요청 B는 `{researcher, visualizer}`로 갈리는지 확인합니다.  
요청 B는 실제 gpt-image-2 이미지 생성이 일어나 1~2분 걸릴 수 있습니다.

In [ ]:
print('=== 요청 A: 조사 + 수치 계산 (차트/이미지 불필요) ===')
result_a = await supervisor_graph.ainvoke({'messages': [{'role': 'user', 'content':
    '전기차 배터리 시장의 최근 동향을 웹에서 간단히 조사하고, '
    '조사된 수치(예: 시장 규모나 성장률)를 바탕으로 간단한 계산을 해서 표로 정리해줘.'}]},
    config={'recursion_limit': 14})
dist_a = worker_dist(result_a)
print(f'\n요청 A 워커 분포: {dict(dist_a)}')

In [ ]:
import koreanize_matplotlib

print('=== 요청 B: 조사 + 차트 + 표지 이미지 ===')
result_b = await supervisor_graph.ainvoke({'messages': [{'role': 'user', 'content':
    '재생에너지(태양광) 시장을 웹에서 아주 간단히 조사하고, '
    '주요 수치 3~4개를 막대 차트로 그려서 outputs/charts/ 에 저장하고, '
    '리포트 표지로 쓸 표지 이미지도 하나 만들어줘.'}]},
    config={'recursion_limit': 16})
dist_b = worker_dist(result_b)
print(f'\n요청 B 워커 분포: {dict(dist_b)}')

In [ ]:
print('=== 성공 기준: 동적 라우팅 + Visualizer의 차트/이미지 구분 ===\n')
print('[기준 1] 요청에 따라 호출되는 전문가 구성이 달라집니다.')
print(f'  요청 A가 부른 전문가: {set(dist_a)}')
print(f'  요청 B가 부른 전문가: {set(dist_b)}')
print(f'  두 구성이 서로 다른가: {set(dist_a) != set(dist_b)} '
      f'(A에 visualizer 없음={"visualizer" not in dist_a}, B에 visualizer 있음={"visualizer" in dist_b})')

charts = glob.glob('outputs/charts/*.png')
images = glob.glob('outputs/generated_images/*.png')
print('\n[기준 2] Visualizer가 데이터 차트와 생성 이미지를 구분해 만들어 냅니다.')
print(f'  outputs/charts/ (python_repl 차트): {len(charts)}개')
print(f'  outputs/generated_images/ (generate_image): {len(images)}개')
print(f'  둘 다 만들어졌는가: {len(charts) > 0 and len(images) > 0}')
print(f'\n요청 B 최종 응답:\n{result_b["messages"][-1].text[:600]}')

---
## 패턴 3: 핸드오프 (사용자 대면 상담 트리아지)

사용자와 여러 턴에 걸쳐 대화하는 도중에 담당 에이전트를 바꿔야 할 때 쓰는 패턴입니다.  
진입 봇 Triage가 첫 메시지의 의도를 분류해 적절한 전문가에게 인계하고, 이후 후속 대화는 그 전문가가 Triage를 거치지 않고 직접 응대합니다.  
경계로는 전체 대화 맥락(messages)이 손실 없이 넘어가고, 여기에 짧은 핸드오프 노트가 더해집니다.

```
사용자 첫 턴 --> [Triage] --Command(goto)--> [Specialist]   (active_agent 저장)
사용자 후속 턴 ------------------------------> [Specialist]   (route_entry가 Triage를 건너뜀)
```

### 3-1. 전문가 전용 도구 + Triage 분류기

In [ ]:
from typing import Literal
from typing_extensions import NotRequired
from pydantic import BaseModel, Field
from langchain.agents import AgentState
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver


# Specialist A(계정/요금) 전용 모의 도구
@tool
def lookup_account(user_id: str) -> str:
    """계정 정보를 조회합니다. user_id: 사용자 ID"""
    db = {
        'u-100': '요금제=Pro, 상태=정상, 다음결제=2026-07-01, 잔여크레딧=12,000',
        'u-200': '요금제=Free, 상태=일시정지, 미납=1건',
    }
    return db.get(user_id, f'{user_id}: 계정을 찾을 수 없습니다.')


@tool
def check_policy(topic: str) -> str:
    """환불/요금 정책을 조회합니다. topic: 'refund' | 'upgrade' | 'cancel'"""
    pol = {
        'refund': '결제 후 7일 이내, 사용량 10% 미만이면 전액 환불.',
        'upgrade': '업그레이드는 즉시 적용, 차액은 일할 계산.',
        'cancel': '해지는 다음 결제일에 적용, 잔여 기간은 계속 사용 가능.',
    }
    return pol.get(topic, f"'{topic}' 정책 항목이 없습니다. (refund/upgrade/cancel)")


# Specialist B(기술 지원) 전용 모의 문서검색 도구 (web_search 대체, 결정적)
@tool
def search_docs(query: str) -> str:
    """제품 기술 문서를 검색합니다. query: 검색어"""
    docs = {
        ('api', 'API'): 'API 키는 대시보드 > 설정 > API에서 발급. 요청 제한 분당 60회.',
        ('webhook', '웹훅'): '웹훅은 설정 > 통합에서 URL 등록. 재시도 최대 5회, 지수 백오프.',
        ('sdk', 'SDK'): 'Python SDK: pip install oursdk. 최소 버전 1.2.',
    }
    q = query.lower()
    for keys, v in docs.items():
        if any(k.lower() in q for k in keys):
            return v
    return '관련 문서를 찾지 못했습니다. (api/webhook/sdk 키워드 지원)'


ACCOUNT_TOOLS = [lookup_account, check_policy]
TECH_TOOLS = [search_docs]

specialist_account = create_agent(
    model=llm, tools=ACCOUNT_TOOLS,
    system_prompt=('당신은 계정/요금 상담 전문가입니다. lookup_account, check_policy 도구로 '
                   '사용자의 계정/결제/환불 문의를 직접 처리합니다. 친절하고 간결하게 답하세요.'),
)
specialist_tech = create_agent(
    model=llm, tools=TECH_TOOLS,
    system_prompt=('당신은 기술 지원 전문가입니다. search_docs 도구로 제품 사용법/연동 문의를 '
                   '직접 처리합니다. 친절하고 간결하게 답하세요.'),
)


class TriageDecision(BaseModel):
    """Triage의 1회성 의도 분류 결과"""
    intent: str = Field(description='사용자 의도 요약')
    target: Literal['account', 'tech'] = Field(description='인계할 전문가')
    note: str = Field(description='전문가에게 넘길 짧은 핸드오프 노트')


TRIAGE_PROMPT = (
    '당신은 상담 진입 봇(Triage)입니다. 사용자의 첫 메시지를 보고 의도를 분류해 '
    '적절한 전문가에게 인계합니다. 계정/결제/환불/요금제 문의는 account, '
    '제품 사용법/API/연동/기술 문의는 tech로 보냅니다. 당신은 직접 답변하지 않습니다.'
)
triage = create_agent(model=llm, tools=[], system_prompt=TRIAGE_PROMPT, response_format=TriageDecision)

print('Triage + Specialist 2종 정의 완료')

### 3-2. 단방향 핸드오프 그래프

In [ ]:
class TriageState(AgentState):
    active_agent: NotRequired[str]


# 어떤 노드가 어떤 턴에 실행됐는지 추적 (Triage가 후속 턴에 다시 도는지 확인용)
RUN_LOG: list[str] = []


def route_entry(state: TriageState) -> Literal['triage', 'specialist_account', 'specialist_tech']:
    """이미 인계된 전문가가 있으면 Triage를 건너뛰고 바로 전문가로 갑니다 (단방향)."""
    active = state.get('active_agent')
    if active == 'account':
        return 'specialist_account'
    if active == 'tech':
        return 'specialist_tech'
    return 'triage'


def triage_node(state: TriageState) -> Command[Literal['specialist_account', 'specialist_tech']]:
    RUN_LOG.append('triage')
    result = triage.invoke({'messages': state['messages']})
    d = result['structured_response']
    target_node = 'specialist_account' if d.target == 'account' else 'specialist_tech'
    print(f"  [Triage] intent='{d.intent}' -> {d.target}")
    # 전체 messages는 체크포인터가 보존합니다. 작은 핸드오프 노트만 덧붙입니다 (Triage 분류 프롬프트는 넘기지 않음).
    note = AIMessage(content=f'[인계] {d.target} 전문가에게 연결합니다. 사유: {d.note}', name='triage')
    return Command(goto=target_node, update={'active_agent': d.target, 'messages': [note]})


def make_specialist_node(name, agent):
    def node(state: TriageState):
        RUN_LOG.append(name)
        # 전문가는 자신의 가벼운 system_prompt와 전용 도구셋만 활성화합니다. Triage 프롬프트는 받지 않습니다.
        r = agent.invoke({'messages': state['messages']})
        return {'messages': [r['messages'][-1]]}
    return node


b3 = StateGraph(TriageState)
b3.add_node('triage', triage_node)
b3.add_node('specialist_account', make_specialist_node('account', specialist_account))
b3.add_node('specialist_tech', make_specialist_node('tech', specialist_tech))
b3.add_conditional_edges(START, route_entry, ['triage', 'specialist_account', 'specialist_tech'])
b3.add_edge('specialist_account', END)
b3.add_edge('specialist_tech', END)
handoff_graph = b3.compile(checkpointer=InMemorySaver())


def run_turn(thread_id, text):
    RUN_LOG.clear()
    cfg = {'configurable': {'thread_id': thread_id}, 'recursion_limit': 12}
    r = handoff_graph.invoke({'messages': [{'role': 'user', 'content': text}]}, config=cfg)
    return list(RUN_LOG), r['messages'][-1].text


handoff_graph

### 3-3. 성공 기준 측정

스레드 1은 계정 의도로 시작해 후속 턴을 2개 더 진행합니다.  
첫 턴만 Triage를 거치고, 후속 턴은 전문가가 직접 처리하는지(`실행 노드`에 triage가 없는지) 확인합니다.

In [ ]:
print('=== 스레드 1: 계정/요금 의도 (멀티턴) ===')
turns_a = [
    '안녕하세요, 제 계정 u-100 상태랑 다음 결제일 알려주세요.',
    '환불 정책은 어떻게 되나요?',
    '그럼 지금 해지하면 남은 기간은 쓸 수 있나요?',
]
for i, t in enumerate(turns_a, 1):
    nodes, reply = run_turn('acc-thread', t)
    print(f'[턴 {i}] 실행 노드={nodes}')
    print(f'        Q: {t}')
    print(f'        A: {reply[:140]}\n')

In [ ]:
print('=== 스레드 2: 기술 의도 (다른 전문가로 라우팅) ===')
turns_b = [
    'API 키는 어디서 발급하나요?',
    '웹훅 재시도 정책도 알려주세요.',
]
for i, t in enumerate(turns_b, 1):
    nodes, reply = run_turn('tech-thread', t)
    print(f'[턴 {i}] 실행 노드={nodes}')
    print(f'        Q: {t}')
    print(f'        A: {reply[:140]}\n')

In [ ]:
print('=== 핸드오프 성공 기준 ===\n')
print('[기준 1] 인계 후 후속 턴은 전문가가 Triage 없이 직접 처리합니다.')
print('  스레드 1에서 턴 1만 triage를 거치고, 턴 2~3은 account 전문가가 단독으로 실행했습니다 (위 실행 노드 참고).')
print('\n[기준 2] 전문가의 활성 도구셋과 프롬프트가 가볍습니다.')
combined = sorted(set(t.name for t in ACCOUNT_TOOLS) | set(t.name for t in TECH_TOOLS))
print(f'  account 전문가 도구: {[t.name for t in ACCOUNT_TOOLS]} ({len(ACCOUNT_TOOLS)}개)')
print(f'  tech 전문가 도구:    {[t.name for t in TECH_TOOLS]} ({len(TECH_TOOLS)}개)')
print(f'  단일 봇이라면 {combined} ({len(combined)}개)에 Triage 분류 프롬프트까지 늘 함께 적재됩니다.')
print('\n[기준 3] 서로 다른 의도는 서로 다른 전문가로 연결됩니다.')
print('  계정 의도는 specialist_account로, 기술 의도는 specialist_tech로 연결됩니다.')

---
## 마무리: 어떤 상황에 어떤 패턴인가

세 패턴은 모두 단일 에이전트의 윈도우와 도구셋을 작게 유지하려는 같은 동기에서 나옵니다.  
무엇을 경계 밖으로 격리하고, 경계로는 무엇만 넘기는지가 다를 뿐입니다.

| 패턴 | 적용 상황 | 경계로 넘기는 것 | 이 노트북에서 본 지표 |
|---|---|---|---|
| 서브에이전트 | 원본 소스가 많아 메인 윈도우에 다 넣기 어려움 | 요약 (원문은 격리) | 소스를 3에서 6으로 늘려도 Main 윈도우 평탄, 모놀리식 대비 크게 작음 |
| 슈퍼바이저 | 성격이 다른 작업을 동적으로 라우팅 | 작업 지시 / 구조화 결과 | 요청별 전문가 부분집합 변화, Visualizer의 차트/이미지 구분 |
| 핸드오프 | 사용자 대면 멀티턴 대화의 소유권 인계 | 전체 대화 맥락(무손실) + 핸드오프 노트 | 후속 턴을 전문가가 직접 처리, 활성 도구셋 경량 |
